# Gaussian Processes — From Scratch

Based on [Peter Roelants' tutorial](https://peterroelants.github.io/posts/gaussian-process-tutorial/)

This notebook covers:
1. Stochastic processes & Brownian motion
2. Gaussian processes: definition and kernel functions
3. Sampling from a GP prior
4. GP regression (noiseless)
5. GP regression with noisy observations
6. Kernel zoo: encoding different priors

In [ ]:
import numpy as np
import scipy.spatial
import scipy.linalg
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

np.random.seed(598)
%matplotlib inline

---
## 1. Stochastic Processes

A **stochastic process** describes systems that evolve randomly over time. Even if the starting point is known, there are many possible trajectories (realizations) the process can take.

**Brownian motion** (Wiener process) is a classic example: a particle suspended in fluid moves randomly due to collisions with other particles. We model this in 1D by taking cumulative steps sampled from:

$$\Delta d \sim \mathcal{N}(0,\, \Delta t)$$

so that the position evolves as:

$$d(t + \Delta t) = d(t) + \Delta d$$

In [ ]:
np.random.seed(598)
# 1D simulation of Brownian motion
total_time = 1
nb_steps = 75
delta_t = total_time / nb_steps
nb_processes = 5
mean = 0.
stdev = np.sqrt(delta_t)

distances = np.cumsum(
    np.random.normal(mean, stdev, (nb_processes, nb_steps)),
    axis=1)

t = np.arange(0, total_time, delta_t)
plt.figure(figsize=(6, 4))
for i in range(nb_processes):
    plt.plot(t, distances[i, :])
plt.title('Brownian motion process\nPosition over time for 5 independent realizations')
plt.xlabel('$t$ (time)', fontsize=13)
plt.ylabel('$d$ (position)', fontsize=13)
plt.xlim([0, 1])
plt.tight_layout()
plt.show()

Each realization is a different function $f(t) = d$. A stochastic process is therefore a **distribution over functions** — we can sample a function from it, and each sampled function may be different.

---
## 2. Gaussian Processes

A **Gaussian process** is a distribution over functions $f(x)$ fully specified by:
- a **mean function** $m(x)$
- a **covariance (kernel) function** $k(x, x')$

$$f(x) \sim \mathcal{GP}\bigl(m(x),\, k(x, x')\bigr)$$

For any finite subset $X = \{\mathbf{x}_1, \ldots, \mathbf{x}_n\}$ of the input domain, the marginal distribution is a multivariate Gaussian:

$$f(X) \sim \mathcal{N}\bigl(m(X),\, k(X, X)\bigr)$$

### The Exponentiated Quadratic (RBF) Kernel

The choice of kernel encodes prior beliefs about the function. The **exponentiated quadratic** kernel (also called squared-exponential or RBF) is:

$$k(x_a, x_b) = \exp\!\left(-\frac{1}{2\sigma^2}\|x_a - x_b\|^2\right)$$

With $\sigma = 1$ this simplifies to $k(x_a, x_b) = \exp\!\left(-\frac{1}{2}\|x_a - x_b\|^2\right)$. Nearby points are highly correlated; distant points are nearly independent.

In [ ]:
def exponentiated_quadratic(xa, xb):
    """Exponentiated quadratic kernel with sigma=1."""
    sq_norm = - scipy.spatial.distance.cdist(xa, xb, 'sqeuclidean') / (2 * 10)
    return np.exp(sq_norm)

In [ ]:
# Visualize the covariance matrix and the kernel as a function of distance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))
xlim = (-3, 3)

# Covariance matrix
X = np.expand_dims(np.linspace(*xlim, 25), 1)
S = exponentiated_quadratic(X, X)
im = ax1.imshow(S, cmap=cm.YlGnBu)
cbar = plt.colorbar(im, ax=ax1, fraction=0.045, pad=0.05)
cbar.ax.set_ylabel('$k(x, x)$', fontsize=10)
ax1.set_title('Exponentiated quadratic\nexample of covariance matrix')
ax1.set_xlabel('x', fontsize=13)
ax1.set_ylabel('x', fontsize=13)
ticks = list(range(xlim[0], xlim[1] + 1))
ax1.set_xticks(np.linspace(0, len(X) - 1, len(ticks)))
ax1.set_yticks(np.linspace(0, len(X) - 1, len(ticks)))
ax1.set_xticklabels(ticks)
ax1.set_yticklabels(ticks)
ax1.grid(False)

# Covariance with x=0
xlim2 = (-4, 4)
X2 = np.expand_dims(np.linspace(*xlim2, 50), 1)
zero = np.array([[0]])
S0 = exponentiated_quadratic(X2, zero)
ax2.plot(X2[:, 0], S0[:, 0], label='$k(x, 0)$')
ax2.set_xlabel('x', fontsize=13)
ax2.set_ylabel('covariance', fontsize=13)
ax2.set_title('Exponentiated quadratic covariance\nbetween $x$ and $0$')
ax2.set_xlim(*xlim2)
ax2.legend(loc=1)

fig.tight_layout()
plt.show()

---
## 3. Sampling from the GP Prior

We cannot sample the full function (infinite domain), but we can sample evaluations at a finite set of points $X$. We draw from the multivariate Gaussian:

$$\mathbf{y} \sim \mathcal{N}\bigl(\mathbf{0},\, k(X, X)\bigr)$$

assuming a zero mean prior, $m(x) = 0$.

In [ ]:
nb_of_samples = 41
number_of_functions = 5

X = np.expand_dims(np.linspace(-4, 4, nb_of_samples), 1)
S = exponentiated_quadratic(X, X)

# Sample from the prior
ys = np.random.multivariate_normal(
    mean=np.zeros(nb_of_samples), cov=S,
    size=number_of_functions)

plt.figure(figsize=(6, 4))
for i in range(number_of_functions):
    plt.plot(X, ys[i], linestyle='-', marker='o', markersize=3)
plt.xlabel('$x$', fontsize=13)
plt.ylabel('$y = f(x)$', fontsize=13)
plt.title('5 function realizations at 41 points\nsampled from a GP with exponentiated quadratic kernel')
plt.xlim([-4, 4])
plt.show()

### 2D Marginal Distributions

For any two input points $x_a, x_b$, the joint distribution of $(y_a, y_b) = (f(x_a), f(x_b))$ is a 2D Gaussian. The shape of this 2D Gaussian is controlled by the kernel:

- **Close points** (e.g. $x = 0$ and $x = 0.2$): high covariance $\Rightarrow$ elongated, correlated ellipse
- **Distant points** (e.g. $x = 0$ and $x = 2$): low covariance $\Rightarrow$ near-circular, uncorrelated

In [ ]:
def generate_surface(mean, covariance, surface_resolution):
    """Generate a 2D density surface for a bivariate Gaussian."""
    x1s = np.linspace(-5, 5, num=surface_resolution)
    x2s = np.linspace(-5, 5, num=surface_resolution)
    x1, x2 = np.meshgrid(x1s, x2s)
    pdf = np.zeros((surface_resolution, surface_resolution))
    for i in range(surface_resolution):
        for j in range(surface_resolution):
            pdf[i, j] = scipy.stats.multivariate_normal.pdf(
                np.array([x1[i, j], x2[i, j]]),
                mean=mean, cov=covariance)
    return x1, x2, pdf

surface_resolution = 50
fig = plt.figure(figsize=(6.2, 3.5))
gs = gridspec.GridSpec(1, 2)
ax_p1 = plt.subplot(gs[0, 0])
ax_p2 = plt.subplot(gs[0, 1], sharex=ax_p1, sharey=ax_p1)

# Strong correlation (nearby points)
X_strong = np.array([[0], [0.2]])
mu = np.array([0., 0.])
S_strong = exponentiated_quadratic(X_strong, X_strong)
y1, y2, p = generate_surface(mu, S_strong, surface_resolution)
con1 = ax_p1.contourf(y1, y2, p, 25, cmap=cm.magma_r)
ax_p1.set_xlabel(f'$y_1 = f(X={X_strong[0,0]})$', fontsize=11, labelpad=0)
ax_p1.set_ylabel(f'$y_2 = f(X={X_strong[1,0]})$', fontsize=11)
ax_p1.axis([-2.7, 2.7, -2.7, 2.7])
ax_p1.set_aspect('equal')
ax_p1.text(-2.3, 2.1,
           f'$k({X_strong[0,0]}, {X_strong[1,0]}) = {S_strong[0,1]:.2f}$',
           fontsize=10)
ax_p1.set_title(f'$X = [{X_strong[0,0]}, {X_strong[1,0]}]$', fontsize=12)
X_0_idx = np.where(np.isclose(X, 0.))[0][0]
X_02_idx = np.where(np.isclose(X, 0.2))[0][0]
y_strong = ys[:, [X_0_idx, X_02_idx]]

for i in range(y_strong.shape[0]):
    ax_p1.plot(y_strong[i, 0], y_strong[i, 1], marker='o')

# Weak correlation (distant points)
X_weak = np.array([[0], [2]])
S_weak = exponentiated_quadratic(X_weak, X_weak)
y1, y2, p = generate_surface(mu, S_weak, surface_resolution)
con2 = ax_p2.contourf(y1, y2, p, 25, cmap=cm.magma_r)
con2.set_cmap(con1.get_cmap())
con2.set_clim(con1.get_clim())
ax_p2.set_xlabel(f'$y_1 = f(X={X_weak[0,0]})$', fontsize=11, labelpad=0)
ax_p2.set_ylabel(f'$y_2 = f(X={X_weak[1,0]})$', fontsize=11)
ax_p2.set_aspect('equal')
ax_p2.text(-2.3, 2.1,
           f'$k({X_weak[0,0]}, {X_weak[1,0]}) = {S_weak[0,1]:.2f}$',
           fontsize=10)
ax_p2.set_title(f'$X = [{X_weak[0,0]}, {X_weak[1,0]}]$', fontsize=12)
divider = make_axes_locatable(ax_p2)
cax = divider.append_axes('right', size='5%', pad=0.02)
cbar = plt.colorbar(con1, ax=ax_p2, cax=cax)
cbar.ax.set_ylabel('density: $p(y_1, y_2)$', fontsize=11)
X_2_idx = np.where(np.isclose(X, 2.))[0][0]
y_weak = ys[:, [X_0_idx, X_2_idx]]

for i in range(y_weak.shape[0]):
    ax_p2.plot(y_weak[i, 0], y_weak[i, 1], marker='o')

fig.suptitle('2D marginal: $y \\sim \\mathcal{N}(0, k(X, X))$')
plt.tight_layout()
plt.show()

---
## 4. GP Regression (Noiseless)

We use the GP as a prior and update it with observations $(X_1, \mathbf{y}_1)$ to get a **posterior** distribution over functions.

Since $\mathbf{y}_1$ (observed) and $\mathbf{y}_2$ (to predict at $X_2$) are jointly Gaussian:

$$\begin{bmatrix} \mathbf{y}_1 \\ \mathbf{y}_2 \end{bmatrix} \sim \mathcal{N}\!\left( \begin{bmatrix} \mu_1 \\ \mu_2 \end{bmatrix},\, \begin{bmatrix} \Sigma_{11} & \Sigma_{12} \\ \Sigma_{21} & \Sigma_{22} \end{bmatrix} \right)$$

where $\Sigma_{11} = k(X_1,X_1)$, $\Sigma_{22} = k(X_2,X_2)$, $\Sigma_{12} = k(X_1,X_2)$.

The **conditional (posterior)** distribution is:

$$p(\mathbf{y}_2 \mid \mathbf{y}_1, X_1, X_2) = \mathcal{N}(\mu_{2|1},\, \Sigma_{2|1})$$

$$\mu_{2|1} = \Sigma_{21}\Sigma_{11}^{-1}\mathbf{y}_1 \qquad (\text{assuming } \mu = 0)$$

$$\Sigma_{2|1} = \Sigma_{22} - \Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12}$$

The posterior mean is a **weighted average of the observations**, where weights come from the kernel.

In [ ]:
def GP(X1, y1, X2, kernel_func):
    """
    Compute the posterior mean and covariance for predictions at X2,
    given observations (X1, y1) and a kernel function.
    """
    S11 = kernel_func(X1, X1)                                # (n1 x n1)
    S12 = kernel_func(X1, X2)                                # (n1 x n2)
    solved = scipy.linalg.solve(S11, S12, assume_a='pos').T  # (n2 x n1)
    mu2 = solved @ y1                                        # posterior mean
    S22 = kernel_func(X2, X2)                                # (n2 x n2)
    S2 = S22 - (solved @ S12)                                # posterior covariance
    return mu2, S2

In [ ]:
np.random.seed(42)

f_sin = lambda x: np.sin(x).flatten()

n1 = 8      # training points
n2 = 75     # test points
ny = 5      # posterior samples to draw
domain = (-6, 6)

X1 = np.random.uniform(domain[0] + 2, domain[1] - 2, size=(n1, 1))
y1 = f_sin(X1)
X2 = np.linspace(domain[0], domain[1], n2).reshape(-1, 1)

mu2, S2 = GP(X1, y1, X2, exponentiated_quadratic)
sigma2 = np.sqrt(np.diag(S2))

y2 = np.random.multivariate_normal(mean=mu2, cov=S2, size=ny)

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(6, 6))

ax1.plot(X2, f_sin(X2), 'b--', label='$\\sin(x)$')
ax1.fill_between(X2.flat, mu2 - 2*sigma2, mu2 + 2*sigma2,
                 color='red', alpha=0.15, label='$2\\sigma_{2|1}$')
ax1.plot(X2, mu2, 'r-', lw=2, label='$\\mu_{2|1}$')
ax1.plot(X1, y1, 'ko', linewidth=2, label='$(x_1, y_1)$')
ax1.set_xlabel('$x$', fontsize=13)
ax1.set_ylabel('$y$', fontsize=13)
ax1.set_title('Posterior distribution and training data (noiseless)')
ax1.axis([domain[0], domain[1], -3, 3])
ax1.legend()

ax2.plot(X2, y2.T, '-')
ax2.set_xlabel('$x$', fontsize=13)
ax2.set_ylabel('$y$', fontsize=13)
ax2.set_title('5 function realizations sampled from the posterior')
ax2.set_xlim(domain)

plt.tight_layout()
plt.show()

Notice the posterior variance **collapses to zero** at the training points — the noiseless GP interpolates the data exactly.

---
## 5. GP Regression with Noisy Observations

Real observations include noise: $f(X_1) = \mathbf{y}_1 + \epsilon$ where $\epsilon \sim \mathcal{N}(0, \sigma_\epsilon^2)$.

We incorporate this by adding the noise variance to the diagonal of the observation kernel:

$$\Sigma_{11} = k(X_1, X_1) + \sigma_\epsilon^2 \, I$$

This allows the posterior to **not** pass exactly through the training points, which is more realistic.

In [ ]:
def GP_noise(X1, y1, X2, kernel_func, sigma_noise):
    """
    Compute the posterior mean and covariance for predictions at X2,
    given noisy observations (X1, y1), a kernel function, and noise std sigma_noise.
    """
    S11 = kernel_func(X1, X1) + (sigma_noise**2) * np.eye(len(X1))
    S12 = kernel_func(X1, X2)
    solved = scipy.linalg.solve(S11, S12, assume_a='pos').T
    mu2 = solved @ y1
    S22 = kernel_func(X2, X2)
    S2 = S22 - (solved @ S12)
    return mu2, S2

In [ ]:
np.random.seed(42)

sigma_noise = 1.0
# Re-use X1 from before; add noise to the observations
y1_noisy = f_sin(X1) + sigma_noise * np.random.randn(n1)

mu2, S2 = GP_noise(X1, y1_noisy, X2, exponentiated_quadratic, sigma_noise)
sigma2 = np.sqrt(np.diag(S2))

y2 = np.random.multivariate_normal(mean=mu2, cov=S2, size=ny)

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(6, 6))

ax1.plot(X2, f_sin(X2), 'b--', label='$\\sin(x)$')
ax1.fill_between(X2.flat, mu2 - 2*sigma2, mu2 + 2*sigma2,
                 color='red', alpha=0.15, label='$2\\sigma_{2|1}$')
ax1.plot(X2, mu2, 'r-', lw=2, label='$\\mu_{2|1}$')
ax1.plot(X1, y1_noisy, 'ko', linewidth=2, label='$(x_1, y_1)$')
ax1.set_xlabel('$x$', fontsize=13)
ax1.set_ylabel('$y$', fontsize=13)
ax1.set_title('Posterior distribution with noisy observations')
ax1.axis([domain[0], domain[1], -3, 3])
ax1.legend()

ax2.plot(X2, y2.T, '-')
ax2.set_xlabel('$x$', fontsize=13)
ax2.set_ylabel('$y$', fontsize=13)
ax2.set_title('5 function realizations sampled from the noisy posterior')
ax2.set_xlim(domain)

plt.tight_layout()
plt.show()

Now the posterior no longer interpolates — it **smooths through** the noisy data. The uncertainty (red band) is non-zero even at the training points.

---
## Summary

| Concept | Key point |
|---|---|
| **Gaussian process** | Distribution over functions, fully specified by $m(x)$ and $k(x,x')$ |
| **Finite marginals** | Any finite evaluation set $\Rightarrow$ multivariate Gaussian |
| **Kernel (RBF)** | Encodes smoothness: nearby inputs are highly correlated |
| **Posterior mean** | Weighted average of observations; weights from the kernel |
| **Noisy GP** | Add $\sigma_\epsilon^2 I$ to $\Sigma_{11}$; posterior no longer interpolates |

> **Note:** Computing $\Sigma_{11}^{-1}\Sigma_{12}$ via `scipy.linalg.solve` (with `assume_a='pos'` to exploit positive-definiteness) is faster and more numerically stable than explicitly inverting $\Sigma_{11}$.

---
## 6. Kernel Zoo: Encoding Different Priors

Based on [Peter Roelants' kernel tutorial](https://peterroelants.github.io/posts/gaussian-process-kernels/)

The kernel is the primary design choice in a GP. Different kernels encode different beliefs about the function's structure. A valid kernel must produce a **positive semi-definite** covariance matrix. Kernels can be **added** (union of properties) or **multiplied** (intersection of properties).

| Kernel | Formula | Prior belief |
|---|---|---|
| **White noise** | $\sigma^2 \, \mathbf{1}[x_a = x_b]$ | i.i.d. noise, no structure |
| **Exponentiated quadratic** | $\sigma^2 \exp\!\left(-\frac{\|x_a-x_b\|^2}{2\ell^2}\right)$ | Infinitely smooth, stationary |
| **Rational quadratic** | $\sigma^2\left(1 + \frac{\|x_a-x_b\|^2}{2\alpha\ell^2}\right)^{-\alpha}$ | Multi-scale smoothness (sum of EQ kernels) |
| **Periodic** | $\sigma^2 \exp\!\left(-\frac{2}{\ell^2}\sin^2\!\left(\pi\frac{|x_a-x_b|}{p}\right)\right)$ | Exactly periodic with period $p$ |
| **Local periodic** | Periodic $\times$ EQ | Periodic locally, decays at distance |

In [ ]:
# Kernel implementations (pure numpy)

def eq_kernel(xa, xb, sigma=1.0, ell=1.0):
    """Exponentiated quadratic with amplitude sigma and lengthscale ell."""
    sq_dist = scipy.spatial.distance.cdist(xa, xb, 'sqeuclidean')
    return (sigma**2) * np.exp(-sq_dist / (2 * ell**2))

def rational_quadratic_kernel(xa, xb, sigma=1.0, ell=1.0, alpha=1.0):
    """Rational quadratic: infinite mixture of EQ kernels at different lengthscales."""
    sq_dist = scipy.spatial.distance.cdist(xa, xb, 'sqeuclidean')
    return (sigma**2) * (1.0 + sq_dist / (2.0 * alpha * ell**2))**(-alpha)

def periodic_kernel(xa, xb, sigma=1.0, ell=1.0, p=1.0):
    """Periodic (ExpSinSquared) kernel with period p."""
    # Use L1 distance for 1D; generalizes to higher D via the sin trick
    dist = scipy.spatial.distance.cdist(xa, xb, 'minkowski', p=1)
    return (sigma**2) * np.exp(-2.0 * np.sin(np.pi * dist / p)**2 / ell**2)

def local_periodic_kernel(xa, xb, sigma=1.0, ell_p=1.0, ell_eq=2.0, p=1.0):
    """Local periodic = periodic * EQ. Periodic locally, fades at long range."""
    dist = scipy.spatial.distance.cdist(xa, xb, 'minkowski', p=1)
    sq_dist = scipy.spatial.distance.cdist(xa, xb, 'sqeuclidean')
    k_per = np.exp(-2.0 * np.sin(np.pi * dist / p)**2 / ell_p**2)
    k_eq = np.exp(-sq_dist / (2.0 * ell_eq**2))
    return (sigma**2) * k_per * k_eq

In [ ]:
# Visualize covariance matrices and sample functions for each kernel

X_vis = np.linspace(-5, 5, 100).reshape(-1, 1)
np.random.seed(0)

kernels = [
    ('Exponentiated\nQuadratic', lambda xa, xb: eq_kernel(xa, xb, ell=1.0)),
    ('Rational\nQuadratic', lambda xa, xb: rational_quadratic_kernel(xa, xb, alpha=1.0)),
    ('Periodic\n(p=2)', lambda xa, xb: periodic_kernel(xa, xb, ell=1.0, p=2.0)),
    ('Local\nPeriodic', lambda xa, xb: local_periodic_kernel(xa, xb, ell_p=1.0, ell_eq=3.0, p=2.0)),
]

fig, axes = plt.subplots(2, len(kernels), figsize=(14, 6))

for col, (name, kern) in enumerate(kernels):
    S = kern(X_vis, X_vis)

    # Top row: covariance matrix
    ax_top = axes[0, col]
    ax_top.imshow(S, cmap=cm.YlGnBu, aspect='auto')
    ax_top.set_title(name, fontsize=11)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.grid(False)
    if col == 0:
        ax_top.set_ylabel('Covariance matrix', fontsize=10)

    # Bottom row: prior samples
    ax_bot = axes[1, col]
    samples = np.random.multivariate_normal(
        mean=np.zeros(len(X_vis)),
        cov=S + 1e-6 * np.eye(len(X_vis)),
        size=4)
    for s in samples:
        ax_bot.plot(X_vis[:, 0], s, lw=1)
    ax_bot.set_xlim([-5, 5])
    ax_bot.set_ylim([-3, 3])
    ax_bot.set_xlabel('$x$', fontsize=10)
    if col == 0:
        ax_bot.set_ylabel('Prior samples $f(x)$', fontsize=10)

fig.suptitle('Kernel comparison: covariance structure and prior function samples', fontsize=12)
plt.tight_layout()
plt.show()

### Composite Kernels

Kernels can be **added** (the function has *either* property) or **multiplied** (the function has *both* properties simultaneously):

- **Sum** $k_1 + k_2$: the function varies according to either structure (logical OR)
- **Product** $k_1 \cdot k_2$: both constraints must hold simultaneously (logical AND)

This is how practitioners build kernels for complex real-world data, such as the classic Mauna Loa CO$_2$ example: long-term trend (EQ) + seasonal cycle (local periodic) + short-term irregularity (rational quadratic) + noise (white).

In [ ]:
# Composite kernel: long-term smooth trend + periodic seasonal component
np.random.seed(1)

def composite_kernel(xa, xb):
    k_trend = eq_kernel(xa, xb, sigma=1.0, ell=3.0)          # slow variation
    k_season = local_periodic_kernel(xa, xb, sigma=0.5,       # periodic locally
                                     ell_p=0.5, ell_eq=4.0, p=1.0)
    k_irregular = rational_quadratic_kernel(xa, xb, sigma=0.3, ell=0.5, alpha=0.5)
    return k_trend + k_season + k_irregular

X_comp = np.linspace(0, 10, 200).reshape(-1, 1)
S_comp = composite_kernel(X_comp, X_comp)
samples_comp = np.random.multivariate_normal(
    mean=np.zeros(len(X_comp)),
    cov=S_comp + 1e-6 * np.eye(len(X_comp)),
    size=4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.imshow(S_comp, cmap=cm.YlGnBu, aspect='auto')
ax1.set_title('Composite kernel\ncovariance matrix')
ax1.set_xticks([]); ax1.set_yticks([]); ax1.grid(False)

for s in samples_comp:
    ax2.plot(X_comp[:, 0], s, lw=1.2)
ax2.set_xlabel('$x$', fontsize=12)
ax2.set_ylabel('$f(x)$', fontsize=12)
ax2.set_title('Prior samples: trend + local periodic + irregular')
ax2.set_xlim([0, 10])

plt.tight_layout()
plt.show()